Using julia and its fast Laplacian solver, we speed up the computation of the resistance score. Tends to work better for large and sparse graphs (for smaller or denser graphs, it might even be slower). Note that julia compiles a function the first time it is called, so the first call might be slower (re-execute same code cell to see the real speedup).

In [1]:
import networkx as nx
import numpy as np
from scipy.sparse import diags

n = 10000
#p = 0.005
m = 2
#G = nx.erdos_renyi_graph(n=n, p=p)
G = nx.barabasi_albert_graph(n=n, m=m)
adj = nx.to_numpy_array(G)
lap = diags(adj.sum(axis=1).flatten()) - adj


In [2]:
from propnetscore.node_selection_fast import NodeSelectionTaskFast
from juliacall import Main as jl

probs = np.random.dirichlet(np.ones(adj.shape[0]))
true_node = np.random.randint(adj.shape[0])
task = NodeSelectionTaskFast(adj, probs, true_node)


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [5]:
import time
task = NodeSelectionTaskFast(adj, probs, true_node)
task._get_laplacian() # warm up, otherwise comparison is unfair

start_time = time.time()
rscore = task.resistance_score()
print("Resistance score (normal) %.5f computed in --- %.3f seconds ---" % (rscore, time.time()-start_time))
start_time = time.time()
rscore_fast = task.resistance_score_fast()
print("Resistance score (fast) %.5f computed in --- %.3f seconds ---" % (rscore_fast, time.time()-start_time))

Resistance score (normal) 0.68399 computed in --- 58.314 seconds ---
 --- Converted adjacency matrix to sparse format in 0.331s
 --- Loaded Julia packages in 0.006s
 --- Transferred sparse adjacency matrix to Julia in 0.001s
 --- Computed resistance score in Julia in 0.008s
Resistance score (fast) 0.68399 computed in --- 0.346 seconds ---


--------
From here onwards, it's about a faster implementation of the diffusion score using Krylov subspace fast computation of the matrix exponential times a vector. It is a bit different, because here we don't compute the full matrix exponential, but only its action on a vector, using the fast expm_multiply function from scipy.

In [4]:
start_time = time.time()
dscore = task.diffusion_score(tau=0.1, convention="positive")
print("Diffusion score (normal) %.5f computed in --- %.3f seconds ---" % (dscore, time.time()-start_time))
start_time = time.time()
dscore_fast = task.diffusion_score_fast(tau=0.1)
print("Diffusion score (fast) %.5f computed in --- %.3f seconds ---" % (dscore_fast, time.time()-start_time))

Diffusion score (normal) 0.45345 computed in --- 25.792 seconds ---
Diffusion score (fast) 0.45345 computed in --- 0.429 seconds ---
